# 05F - Class Imbalance Handling

Explore and address class imbalance using common resampling techniques.

In [1]:

import pandas as pd
from sklearn.model_selection import train_test_split

df=pd.read_csv(r"/mnt/data/american_bankruptcy.csv")
df['target']=df['status_label'].map({'alive':0,'failed':1})
drop=['status_label','target']
if 'company_name' in df.columns:
    drop.append('company_name')
X=df.drop(columns=drop)
X=X.select_dtypes(include='number')
y=df['target']

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y)


## Original Class Distribution

In [2]:

dist=y_train.value_counts().rename_axis('Class').reset_index(name='Count')
dist['Percentage']=(dist['Count']/dist['Count'].sum()*100).round(2)
display(dist)


,Class,Count,Percentage
0,0,58769,93.37
1,1,4176,6.63


## Random Oversampling

In [3]:

try:
    from imblearn.over_sampling import RandomOverSampler
    ros=RandomOverSampler(random_state=42)
    X_ros,y_ros=ros.fit_resample(X_train,y_train)
    print("After Random Oversampling:")
    print(y_ros.value_counts())
except Exception as e:
    print("imbalanced-learn not available:",e)


After Random Oversampling:
target
0    58769
1    58769
Name: count, dtype: int64


## SMOTE

In [4]:

try:
    from imblearn.over_sampling import SMOTE
    sm=SMOTE(random_state=42)
    X_sm,y_sm=sm.fit_resample(X_train,y_train)
    print("After SMOTE:")
    print(y_sm.value_counts())
except Exception as e:
    print("SMOTE unavailable:",e)


After SMOTE:
target
0    58769
1    58769
Name: count, dtype: int64


## Random Undersampling

In [5]:

try:
    from imblearn.under_sampling import RandomUnderSampler
    rus=RandomUnderSampler(random_state=42)
    X_rus,y_rus=rus.fit_resample(X_train,y_train)
    print("After Random Undersampling:")
    print(y_rus.value_counts())
except Exception as e:
    print("RandomUnderSampler unavailable:",e)


After Random Undersampling:
target
0    4176
1    4176
Name: count, dtype: int64


## Class Weight Strategy

In [6]:

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

weights=compute_class_weight(class_weight='balanced',
                             classes=np.unique(y_train),
                             y=y_train)
cw=dict(zip(np.unique(y_train),weights))
print("Suggested class weights:")
print(cw)


Suggested class weights:
{np.int64(0): np.float64(0.5355289353230445), np.int64(1): np.float64(7.536518199233717)}


## Recommendations

In [7]:

tips=[
"Use SMOTE only on the training set.",
"Never oversample validation or test data.",
"For tree ensembles, compare SMOTE with class_weight='balanced'.",
"Evaluate Recall, F1-score, ROC-AUC and PR-AUC instead of accuracy alone."
]
for i,t in enumerate(tips,1):
    print(f"{i}. {t}")


1. Use SMOTE only on the training set.
2. Never oversample validation or test data.
3. For tree ensembles, compare SMOTE with class_weight='balanced'.
4. Evaluate Recall, F1-score, ROC-AUC and PR-AUC instead of accuracy alone.
